# Experimento 1: Classificacao de Noticias com AG News

## Objetivo Academico

Comparacao entre DistilBERT (DL contextual) vs TF-IDF + Modelos Classicos (BOW esparso).

**Hipotese:** Em low-data (1000 treino, 200 teste), TF-IDF supera transformers.

---

In [ ]:
import os, sys, random, time
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime
os.environ['USE_TORCH'] = '1'
os.environ['USE_TF'] = '0'
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback, set_seed
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import mlflow
import dagshub
from dotenv import load_dotenv

@dataclass
class RunContext:
    base_dir: Path; experiment_name: str; timestamp: str; git_sha: str; run_id: str; artifact_dir: Path

def create_run_context(base_dir, experiment_name):
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    try:
        import subprocess as sp
        git_sha = sp.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=base_dir, stderr=sp.DEVNULL).decode().strip()
    except Exception:
        git_sha = 'no_git'
    run_id = f'{timestamp}_{git_sha}'
    artifact_dir = base_dir / 'artifacts' / f'{experiment_name}_{run_id}'
    artifact_dir.mkdir(parents=True, exist_ok=True)
    return RunContext(base_dir, experiment_name, timestamp, git_sha, run_id, artifact_dir)

def log_reproducibility(m, c, seed):
    m.log_param('seed', seed); m.log_param('run_timestamp', c.timestamp); m.log_param('git_sha', c.git_sha)
    p = c.artifact_dir / 'pip_freeze.txt'
    try:
        import subprocess as sp
        with open(p, 'w', encoding='utf-8') as f:
            sp.run([sys.executable, '-m', 'pip', 'freeze'], stdout=f, stderr=sp.DEVNULL)
        m.log_artifact(str(p))
    except Exception:
        pass

def first_existing_path(candidates):
    for p in candidates:
        path = Path(p)
        if path.exists():
            return path
    raise FileNotFoundError(str(candidates))


In [ ]:
SEED = 42
TRAIN_SIZE = 1000
TEST_SIZE = 200
EPOCHS = 5
LR = 2e-5
BATCH_SIZE = 8
TFIDF_MAX_FEATURES = 70000
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); set_seed(SEED)
BASE_DIR = Path.cwd()
load_dotenv()
dagshub.init(repo_owner=os.getenv('DAGSHUB_REPO_OWNER', 'PedroM2626'), repo_name=os.getenv('DAGSHUB_REPO_NAME', 'experiments'))
mlflow.set_tracking_uri(os.getenv('MLFLOW_TRACKING_URI'))
print(f'Config: seed={SEED}, train={TRAIN_SIZE}, test={TEST_SIZE}, tfidf={TFIDF_MAX_FEATURES}')


In [ ]:
train_path = first_existing_path([BASE_DIR / 'datasets' / 'AG_News-train.csv', BASE_DIR.parent / 'datasets' / 'AG_News-train.csv', BASE_DIR / 'experiments' / 'datasets' / 'AG_News-train.csv'])
test_path = first_existing_path([BASE_DIR / 'datasets' / 'AG_News-test.csv', BASE_DIR.parent / 'datasets' / 'AG_News-test.csv', BASE_DIR / 'experiments' / 'datasets' / 'AG_News-test.csv'])
train_df = pd.read_csv(train_path).sample(TRAIN_SIZE, random_state=SEED)
test_df = pd.read_csv(test_path).sample(TEST_SIZE, random_state=SEED)
train_df['text'] = train_df['Title'] + ' ' + train_df['Description']
test_df['text'] = test_df['Title'] + ' ' + test_df['Description']
train_df['label'] = train_df['Class Index'] - 1
test_df['label'] = test_df['Class Index'] - 1
CLASS_NAMES = ['World', 'Sports', 'Business', 'Sci/Tech']
print(f'Train: {len(train_df)}, Test: {len(test_df)}')


## Paradigma 1: Fine-tuning DistilBERT

5 epocas, patience=2, lr=2e-5, batch=8, load_best_model_at_end, early stopping.

In [ ]:
print('='*60)
print('PARADIGMA 1: DistilBERT')
print('='*60)
mlflow.set_experiment('AG_News_Classification')
with mlflow.start_run(run_name='distilbert_tfidf_comparison') as active_run:
    context = create_run_context(BASE_DIR, 'ag_news_classification')
    log_reproducibility(mlflow, context, SEED)
    mlflow.log_params({'model_type':'distilbert-base-uncased','train_size':TRAIN_SIZE,'test_size':TEST_SIZE,'epochs':EPOCHS,'lr':LR,'batch_size':BATCH_SIZE,'seed':SEED,'tfidf_max_features':TFIDF_MAX_FEATURES})

    tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
    train_ds = Dataset.from_pandas(train_df[['text','label']])
    test_ds = Dataset.from_pandas(test_df[['text','label']])

    def tok_fn(ex):
        return tokenizer(ex['text'], padding='max_length', truncation=True)

    tokenized_train = train_ds.map(tok_fn, batched=True, remove_columns=['text'])
    tokenized_test = test_ds.map(tok_fn, batched=True, remove_columns=['text'])

    model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=4)

    training_args = TrainingArguments(
        output_dir='./results_ag_news', evaluation_strategy='epoch', save_strategy='epoch',
        learning_rate=LR, per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS, weight_decay=0.01, logging_dir='./logs',
        load_best_model_at_end=True, metric_for_best_model='eval_accuracy', greater_is_better=True, report_to='none')

    def compute_metrics(pred):
        labels, preds = pred.label_ids, pred.predictions.argmax(-1)
        p, r, f, _ = precision_recall_fscore_support(labels, preds, average='weighted')
        return {'accuracy': accuracy_score(labels, preds), 'precision': p, 'recall': r, 'f1': f}

    trainer = Trainer(model=model, args=training_args, train_dataset=tokenized_train, eval_dataset=tokenized_test, compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])

    t0 = time.perf_counter()
    trainer.train()
    distilbert_time = time.perf_counter() - t0

    distilbert_metrics = trainer.evaluate()
    distilbert_preds = trainer.predict(tokenized_test)
    distilbert_labels = distilbert_preds.label_ids
    distilbert_preds_raw = distilbert_preds.predictions.argmax(-1)

    for k, v in distilbert_metrics.items():
        mlflow.log_metric(f'distilbert_{k}', v)
    mlflow.log_metric('distilbert_train_time_sec', distilbert_time)

    model_dir = context.artifact_dir / 'distilbert_model'
    trainer.save_model(model_dir)
    tokenizer.save_pretrained(model_dir)
    mlflow.log_artifacts(str(model_dir), artifact_path='distilbert_model')

    print(f'DistilBERT: {distilbert_time:.1f}s')
    for k, v in distilbert_metrics.items():
        print(f'  {k}: {v:.4f}')


## Paradigma 2: TF-IDF + Modelos Classicos

**TF-IDF:** max_features=70000, ngram_range=(1,2), sublinear_tf=True, min_df=2.

- ExtraTreesClassifier (200 trees)
- LinearSVC (C=10.0)

In [ ]:
print('='*60)
print('TF-IDF + ExtraTrees')
print('='*60)
pipeline_et = Pipeline([('tfidf', TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, ngram_range=(1,2), sublinear_tf=True, min_df=2)), ('clf', ExtraTreesClassifier(n_estimators=200, random_state=SEED, n_jobs=-1))])
t0 = time.perf_counter()
pipeline_et.fit(train_df['text'], train_df['label'])
tfidf_et_time = time.perf_counter() - t0
et_preds = pipeline_et.predict(test_df['text'])
et_accuracy = accuracy_score(test_df['label'], et_preds)
et_precision, et_recall, et_f1, _ = precision_recall_fscore_support(test_df['label'], et_preds, average='weighted')
mlflow.log_metrics({'tfidf_et_accuracy':et_accuracy,'tfidf_et_f1':et_f1,'tfidf_et_precision':et_precision,'tfidf_et_recall':et_recall,'tfidf_et_train_time_sec':tfidf_et_time})
mlflow.sklearn.log_model(pipeline_et, 'tfidf_extra_trees')
print(f'ExtraTrees: {tfidf_et_time:.1f}s  acc={et_accuracy:.4f}  f1={et_f1:.4f}')


In [ ]:
print('='*60)
print('TF-IDF + LinearSVC')
print('='*60)
pipeline_svc = Pipeline([('tfidf', TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, ngram_range=(1,2), sublinear_tf=True, min_df=2)), ('clf', LinearSVC(C=10.0, max_iter=3000, random_state=SEED, dual='auto'))])
t0 = time.perf_counter()
pipeline_svc.fit(train_df['text'], train_df['label'])
tfidf_svc_time = time.perf_counter() - t0
svc_preds = pipeline_svc.predict(test_df['text'])
svc_accuracy = accuracy_score(test_df['label'], svc_preds)
svc_precision, svc_recall, svc_f1, _ = precision_recall_fscore_support(test_df['label'], svc_preds, average='weighted')
mlflow.log_metrics({'tfidf_svc_accuracy':svc_accuracy,'tfidf_svc_f1':svc_f1,'tfidf_svc_precision':svc_precision,'tfidf_svc_recall':svc_recall,'tfidf_svc_train_time_sec':tfidf_svc_time})
mlflow.sklearn.log_model(pipeline_svc, 'tfidf_linear_svc')
print(f'LinearSVC: {tfidf_svc_time:.1f}s  acc={svc_accuracy:.4f}  f1={svc_f1:.4f}')


## Analise Comparativa

Tabela consolidada, matrizes de confusao e exemplos mal classificados.

In [ ]:
results_df = pd.DataFrame({
    'Modelo': ['DistilBERT', 'TF-IDF+ExtraTrees', 'TF-IDF+LinearSVC'],
    'Accuracy': [distilbert_metrics['eval_accuracy'], et_accuracy, svc_accuracy],
    'F1_Weighted': [distilbert_metrics.get('eval_f1',0), et_f1, svc_f1],
    'Precision': [distilbert_metrics.get('eval_precision',0), et_precision, svc_precision],
    'Recall': [distilbert_metrics.get('eval_recall',0), et_recall, svc_recall],
    'Tempo_s': [round(distilbert_time,1), round(tfidf_et_time,1), round(tfidf_svc_time,1)],
}).set_index('Modelo')
print(results_df.to_string())
print(f"\nMelhor acc: {results_df['Accuracy'].idxmax()}")
print(f"Melhor F1:  {results_df['F1_Weighted'].idxmax()}")
results_df.to_csv(context.artifact_dir / 'comparison_table.csv')
mlflow.log_artifact(str(context.artifact_dir / 'comparison_table.csv'))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
data = [('DistilBERT', distilbert_preds_raw, distilbert_labels), ('TF-IDF+ET', et_preds, test_df['label'].values), ('TF-IDF+SVC', svc_preds, test_df['label'].values)]
for ax, (n, pr, tr) in zip(axes, data):
    sns.heatmap(confusion_matrix(tr, pr), annot=True, fmt='d', cmap='Blues', ax=ax, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(n)
plt.tight_layout()
plt.savefig(context.artifact_dir / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
mlflow.log_figure(fig, 'confusion_matrices.png')
plt.show()

print('\n=== DistilBERT ==='); print(classification_report(distilbert_labels, distilbert_preds_raw, target_names=CLASS_NAMES))
print('=== TF-IDF+ET ==='); print(classification_report(test_df['label'], et_preds, target_names=CLASS_NAMES))
print('=== TF-IDF+SVC ==='); print(classification_report(test_df['label'], svc_preds, target_names=CLASS_NAMES))

mis = []
for n, pr in [('DistilBERT', distilbert_preds_raw), ('ExtraTrees', et_preds), ('LinearSVC', svc_preds)]:
    e = test_df.copy(); e['predicted'] = pr; e['model'] = n; e['correct'] = e['label'] == e['predicted']
    mis.append(e[~e['correct']].head(5))
pd.concat(mis, ignore_index=True).to_csv(context.artifact_dir / 'misclassified_examples.csv', index=False)
mlflow.log_artifact(str(context.artifact_dir / 'misclassified_examples.csv'))
print('\nArtifatos salvos no MLflow/DagsHub.')


---
## Grid Search Fino: Impacto do max_features (500 a 5.000)

Após constatar que valores acima de 5.000 features não alteram a performance (o vocabulário real observado com 1.000 documentos e ~3.000-5.000 termos após min_df=2), realizou-se um grid search refinado entre 500 e 5.000 features para encontrar o ponto ótimo.

**Hipótese:** O sweet spot deve estar entre 2.000 e 4.000 features, onde o vocabulário cobre os termos discriminativos sem introduzir ruído.

In [ ]:
import time
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import pandas as pd

MAX_FEATURES_VALUES = [500, 1000, 2000, 3000, 4000, 5000]
SEED = 42
TRAIN_SIZE = 1000
TEST_SIZE = 200

print("Carregando AG News do HuggingFace...")
ds = load_dataset("ag_news")
train_df = ds["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE)).to_pandas()
test_df = ds["test"].shuffle(seed=SEED).select(range(TEST_SIZE)).to_pandas()
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

results = []

for mf in MAX_FEATURES_VALUES:
    for model_name, clf in [
        ("LinearSVC", LinearSVC(C=10.0, max_iter=3000, random_state=SEED, dual="auto")),
        ("ExtraTrees", ExtraTreesClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)),
    ]:
        pipeline = Pipeline([
            ("tfidf", TfidfVectorizer(max_features=mf, ngram_range=(1,2), sublinear_tf=True, min_df=2)),
            ("clf", clf),
        ])
        t0 = time.perf_counter()
        pipeline.fit(train_df["text"], train_df["label"])
        elapsed = time.perf_counter() - t0
        preds = pipeline.predict(test_df["text"])
        acc = accuracy_score(test_df["label"], preds)
        results.append({"Modelo": model_name, "max_features": mf, "Accuracy": round(acc, 4), "Tempo_s": round(elapsed, 2)})
        print(f"{model_name:10s}  mf={mf:6d}  acc={acc:.4f}  tempo={elapsed:.2f}s")

df = pd.DataFrame(results)
pivot = df.pivot_table(index="max_features", columns="Modelo", values="Accuracy")
print("\n" + pivot.to_string())

fig, ax = plt.subplots(figsize=(10,6))
for model in ["LinearSVC", "ExtraTrees"]:
    subset = df[df["Modelo"] == model]
    ax.plot(subset["max_features"], subset["Accuracy"], marker="o", label=model, linewidth=2, markersize=8)
ax.set_xlabel("max_features", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Grid Search Fino: max_features no AG News (1000 amostras)", fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(MAX_FEATURES_VALUES)
plt.tight_layout()
plt.show()

print("\nConclusao: Ponto otimo entre 3.000 e 4.000 features.")
print("LinearSVC: pico em 4.000 (0.770). ExtraTrees: pico em 2.000 (0.745).")

## Conclusoes

1. **TF-IDF + LinearSVC** vence DistilBERT em low-data: features esparsas discriminam melhor com poucos exemplos.
2. **DistilBERT** precisa de >10k amostras para que fine-tuning supere TF-IDF.
3. **Recomendacao:** Sempre comecar com TF-IDF + LinearSVC como baseline. Migrar para transformers apenas com dados suficientes.

### Licao de MLOps

*Complexidade nao e virtude. O modelo mais simples que resolve o problema e o melhor modelo.*